In [ ]:
## 第一步：挂载 Google Drive

from google.colab import drive

drive.mount('/content/drive')

import os

# 确认 Drive 挂载成功
print('Drive 已挂载:', os.path.exists('/content/drive/MyDrive'))

## 第二步：克隆代码仓库

将下方
`GITHUB_REPO`
替换为你自己的仓库地址（`https: // github.com / < 你的用户名 >/ RecON.git`）。

GITHUB_REPO = 'https://github.com/Woomessi/Trackerless_3D_Ultrasound_Reconstruction.git'  # ← 修改此处
BRANCH = 'main'  # 如使用其他分支请修改
PROJECT_DIR = '/content/Trackerless_3D_Ultrasound_Reconstruction'

if not os.path.exists(PROJECT_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO} {PROJECT_DIR}
else:
    print('目录已存在，执行 git pull 更新...')
    !git -C {PROJECT_DIR} pull

%cd {PROJECT_DIR}
print('当前目录:', os.getcwd())

## 第三步：安装依赖


# 确认 PyTorch 版本及 CUDA 可用性
import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
# 安装额外依赖（timm、pyvista、h5py；opencv/scipy/numpy 已预装）
!pip install timm pyvista h5py --quiet

# 验证所有依赖可正常导入
import importlib

for pkg in ['cv2', 'numpy', 'scipy', 'timm', 'h5py', 'pyvista']:
    try:
        importlib.import_module(pkg)
        print(f'  ✓ {pkg}')
    except ImportError as e:
        print(f'  ✗ {pkg}: {e}')

## 第四步：挂载数据


DRIVE_DATA_DIR = '/content/drive/MyDrive/projects/3D_US_REC/datasets'  # ← 若 Drive 中路径不同请修改

# 检查 Drive 中数据是否存在
assert os.path.exists(DRIVE_DATA_DIR), f'未找到 Drive 数据目录: {DRIVE_DATA_DIR}'
assert os.path.exists(os.path.join(DRIVE_DATA_DIR, 'frames_transfs')), '缺少 frames_transfs 目录'
assert os.path.exists(os.path.join(DRIVE_DATA_DIR, 'calib_matrix.csv')), '缺少 calib_matrix.csv'

# 用软链接将 Drive 数据映射到项目的 data/ 目录（无需复制，节省空间和时间）
local_data_dir = os.path.join(PROJECT_DIR, 'data')
os.makedirs(local_data_dir, exist_ok=True)


def make_symlink(src, dst):
    if os.path.lexists(dst):
        os.remove(dst)
    os.symlink(src, dst)
    print(f'链接: {dst} → {src}')


make_symlink(
    os.path.join(DRIVE_DATA_DIR, 'frames_transfs'),
    os.path.join(local_data_dir, 'frames_transfs')
)
make_symlink(
    os.path.join(DRIVE_DATA_DIR, 'calib_matrix.csv'),
    os.path.join(local_data_dir, 'calib_matrix.csv')
)

# 验证数据可读
import h5py, glob

h5_files = glob.glob(os.path.join(local_data_dir, 'frames_transfs', '**', '*.h5'), recursive=True)
print(f'\n找到 {len(h5_files)} 个 h5 文件')
with h5py.File(h5_files[0], 'r') as f:
    print(f'示例文件: {h5_files[0]}')
    print(f'  键: {list(f.keys())}')
    print(f'  frames shape: {f["frames"].shape}')
    print(f'  tforms shape: {f["tforms"].shape}')

## 第五步：配置模型保存路径


DRIVE_SAVE_DIR = '/content/drive/MyDrive/projects/3D_US_REC/save'  # ← 可自定义
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

local_save_dir = os.path.join(PROJECT_DIR, 'save')
make_symlink(DRIVE_SAVE_DIR, local_save_dir)

print(f'检查点将保存至: {DRIVE_SAVE_DIR}')

## 第七步：Fine-tuning 训练（scan-level 自监督）

以下
cells
运行
`trial / train / main_complete_finetuning_train.py`，对预训练的
baseline
backbone
进行自监督
fine - tuning。

** 前置条件： **
1.
已完成第一步～第五步（Drive
挂载、代码克隆、依赖安装、save / 软链接）
2.
Drive
中已有预训练
baseline
checkpoint（`RecON_save / online_baseline_bk - hp_bk - TUS_complete / online_baseline_bk_backbone_230.pth`）
3.
TUS
完整数据集（`train_part1 / ` 目录，含各
subject
子目录及
`.h5
` 文件）已上传至
Drive

!git -C {PROJECT_DIR} pull

import json, os

PROJECT_DIR = '/content/Trackerless_3D_Ultrasound_Reconstruction'

# ── 配置：将下方路径修改为 Drive 中 TUS 数据集的实际位置 ─────────────────────
# 数据目录结构: DRIVE_TUS_DIR/<subject>/<scan>.h5
DRIVE_TUS_DIR = '/content/drive/MyDrive/projects/3D_US_REC/datasets/frames_transfs'  # ← 修改此处

assert os.path.exists(DRIVE_TUS_DIR), (
    f'TUS 数据目录未找到: {DRIVE_TUS_DIR}\n'
    '请将 TUS 数据集上传到 Drive 后修改 DRIVE_TUS_DIR。'
)

# 统计 h5 文件数量
import glob

h5_files = glob.glob(os.path.join(DRIVE_TUS_DIR, '**', '*.h5'), recursive=True)
print(f'找到 {len(h5_files)} 个 .h5 文件')
assert h5_files, '目录下没有 .h5 文件，请检查路径和数据结构。'

# 更新 TUS_complete_scan.json 中的数据路径
cfg_path = os.path.join(PROJECT_DIR, 'res/datasets/TUS_complete_scan.json')
with open(cfg_path) as f:
    cfg = json.load(f)
cfg['paths']['h5'] = DRIVE_TUS_DIR
with open(cfg_path, 'w') as f:
    json.dump(cfg, f, indent=2)
print(f'已更新 TUS_complete_scan.json -> h5: {DRIVE_TUS_DIR}')

# 检查预训练 checkpoint 是否存在
ckpt_path = os.path.join(
    PROJECT_DIR,
    'save/online_baseline_bk-hp_bk-TUS_complete/online_baseline_bk_backbone_230.pth'
)
if not os.path.exists(ckpt_path):
    print(f'\n[警告] 预训练 checkpoint 未找到: {ckpt_path}')
    print('请确认 Drive 中 RecON_save/ 目录包含对应文件，且 save/ 软链接已建立（第五步）。')
else:
    import os.path as osp

    print(f'预训练 checkpoint: {osp.getsize(ckpt_path) / 1e6:.1f} MB  ✓')

%cd /content/Trackerless_3D_Ultrasound_Reconstruction

# 启动 fine-tuning（训练轮数和保存间隔由 res/run/hp_finetune_bk.json 控制）
# !python trial/train/main_complete_finetuning_train.py
!PYTHONPATH=/content/Trackerless_3D_Ultrasound_Reconstruction python trial/train/main_complete_finetuning_train.py